In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv("final_cleaned_dataset.csv")

print("Dataset Shape:", df.shape)
df.head()


Dataset Shape: (84950, 2)


,news,label
0,law enforcement high alert following threat co...,1
1,post vote hillary already,1
2,unbelievable obama attorney general say charlo...,1
3,bobby jindal raised hindu us story christian c...,0
4,satan russia unvelis image terrifying new supe...,1


In [3]:
# Basic EDA - Missing values and data types
print("Data Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Data Types:
news     object
label     int64
dtype: object

Missing Values:
news     0
label    0
dtype: int64


In [4]:
# Basic EDA - Target distribution
print("Label Distribution:")
print(df['label'].value_counts())
print("\nLabel Percentage:")
print(df['label'].value_counts(normalize=True)*100)

Label Distribution:
label
1    45198
0    39752
Name: count, dtype: int64

Label Percentage:
label
1    53.205415
0    46.794585
Name: proportion, dtype: float64


In [5]:
# Basic EDA - Text length analysis
df['char_count'] = df['news'].astype(str).apply(len)
df['word_count'] = df['news'].astype(str).apply(lambda x: len(x.split()))

print(df[['char_count','word_count']].describe())

         char_count    word_count
count  84950.000000  84950.000000
mean    1702.645650    236.457340
std     2255.512609    314.152833
min        1.000000      1.000000
25%       80.000000     12.000000
50%     1285.000000    180.000000
75%     2291.000000    317.000000
max    79567.000000  12007.000000


In [6]:
# Basic EDA - Duplicate rows
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [7]:

X = df["news"]
y = df["label"]

In [8]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)



Train Shape: (67960,)
Test Shape : (16990,)


In [9]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vectorization Completed")

Vectorization Completed


In [10]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        solver="liblinear"
    ),

    "Naive Bayes": MultinomialNB(),

    "Linear SVM": LinearSVC()
}


In [11]:
results = []

for name, model in models.items():

    model.fit(X_train_vec, y_train)

    y_pred = model.predict(X_test_vec)

    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, average="weighted")
    rec = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    results.append([
        name,
        round(acc, 4),
        round(pre, 4),
        round(rec, 4),
        round(f1, 4)
    ])

print("Training Completed")


Training Completed


In [12]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print(results_df)


                 Model  Accuracy  Precision  Recall  F1 Score
0           Linear SVM    0.9057     0.9063  0.9057    0.9055
1  Logistic Regression    0.9012     0.9024  0.9012    0.9009
2          Naive Bayes    0.8183     0.8225  0.8183    0.8167


In [13]:
best_model = results_df.iloc[0]

print("\nBest Performing Model:")
print(best_model)


Best Performing Model:
Model        Linear SVM
Accuracy         0.9057
Precision        0.9063
Recall           0.9057
F1 Score         0.9055
Name: 0, dtype: object
